# Week 08 — Power BI Foundation and Page 1

**P15 PropIQ | ZENAIZ × BVRIT Hyderabad**

This notebook is for the Week 08 deliverable `notebooks/06_powerbi_export.ipynb`.

## Week outcome
Export validated Gold data, create a safe Power BI model, and build the **Market Overview** page.

## Source requirements used
- Export **only validated Gold objects**.
- Use **one-to-many relationships** with deliberate filter direction.
- Build KPI cards and monthly trends.
- Reconcile selected Power BI visuals back to Gold.
- Record model and refresh evidence.

> This notebook does not invent expected KPI totals. It calculates checks dynamically from the current Gold tables.


## 1. Before running

Week 08 depends on the accepted Week 07 Gold layer.

Expected Gold object groups from the playbook:

**Dimensions**
`gold_dim_date`, `gold_dim_locality`, `gold_dim_property`, `gold_dim_broker`, `gold_dim_listing_status`, `gold_dim_price_band`, `gold_dim_lead_channel`

**Facts**
`gold_fact_listing`, `gold_fact_lead`

**Summaries**
`gold_locality_price_summary`, `gold_listing_performance_summary`, `gold_lead_conversion_summary`, `gold_broker_performance_summary`, `gold_inventory_age_summary`

Set the catalog and schema below to the catalog/schema where your Week 07 Gold tables exist.


**PropIQ environment used:** `workspace.default`. The notebook checks the 14 Gold objects currently present in that catalog/schema.


In [0]:
# Databricks setup
# Replace these two values with the catalog/schema used by your Week 07 Gold layer.

CATALOG = "workspace"
SCHEMA = "default"

assert CATALOG == "workspace", "Use the PropIQ catalog: workspace."
assert SCHEMA == "default", "Use the PropIQ schema: default."

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print(f"Using Gold layer: {CATALOG}.{SCHEMA}")


Using Gold layer: workspace.default


## 2. Confirm the Gold objects exist

This is a pre-export gate. Do not export a table that is missing or replace a missing Gold object with a lower-layer table.


In [0]:
gold_objects = [
    "gold_dim_date",
    "gold_dim_locality",
    "gold_dim_property",
    "gold_dim_broker",
    "gold_dim_listing_status",
    "gold_dim_price_band",
    "gold_dim_lead_channel",
    "gold_fact_listing",
    "gold_fact_lead",
    "gold_locality_price_summary",
    "gold_listing_performance_summary",
    "gold_lead_conversion_summary",
    "gold_broker_performance_summary",
    "gold_inventory_age_summary",
]

existing = {
    r["tableName"]
    for r in spark.sql("SHOW TABLES").collect()
    if r["tableName"] in gold_objects
}

missing = sorted(set(gold_objects) - existing)
print("Existing Gold objects:")
for name in sorted(existing):
    print("  ✓", name)

if missing:
    raise ValueError("Missing expected Gold objects: " + ", ".join(missing))

print("\nGold object gate PASSED.")


Existing Gold objects:
  ✓ gold_broker_performance_summary
  ✓ gold_dim_broker
  ✓ gold_dim_date
  ✓ gold_dim_lead_channel
  ✓ gold_dim_listing_status
  ✓ gold_dim_locality
  ✓ gold_dim_price_band
  ✓ gold_dim_property
  ✓ gold_fact_lead
  ✓ gold_fact_listing
  ✓ gold_inventory_age_summary
  ✓ gold_lead_conversion_summary
  ✓ gold_listing_performance_summary
  ✓ gold_locality_price_summary

Gold object gate PASSED.


## 3. Inspect grain and schema before export

The Week 08 failure scenario is a many-to-many Power BI relationship that causes totals to change with slicers. Inspect the keys and schema before loading the data into Power BI.


In [0]:
# Review the main fact schemas available in PropIQ Gold

fact_tables = [
    "gold_fact_listing",
    "gold_fact_lead"
]

for table_name in fact_tables:

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    df = spark.table(
        f"{CATALOG}.{SCHEMA}.{table_name}"
    )

    df.printSchema()

    print("Rows:", df.count())


gold_fact_listing
root
 |-- record_uid: string (nullable = true)
 |-- listing_id: string (nullable = true)
 |-- locality_id: string (nullable = true)
 |-- broker_id: string (nullable = true)
 |-- property_key: integer (nullable = true)
 |-- price_band_key: integer (nullable = true)
 |-- listing_status: string (nullable = true)
 |-- asking_price_inr: long (nullable = true)
 |-- built_up_area_sqft: integer (nullable = true)
 |-- price_per_sqft: long (nullable = true)
 |-- calculated_price_per_sqft: long (nullable = true)
 |-- price_per_sqft_variance: long (nullable = true)
 |-- listing_created_date: date (nullable = true)
 |-- completion_date: date (nullable = true)
 |-- last_updated_timestamp: timestamp (nullable = true)
 |-- is_completed: boolean (nullable = true)
 |-- is_chronology_valid: boolean (nullable = true)
 |-- actual_days_on_market: integer (nullable = true)
 |-- days_since_last_update: integer (nullable = true)
 |-- days_on_market_status_aware: integer (nullable = true)
 |-

In [0]:
# Basic grain checks

checks = {
    "gold_fact_listing": ["listing_id"],
    "gold_fact_lead": ["lead_id"]
}

for table_name, keys in checks.items():

    df = spark.table(
        f"{CATALOG}.{SCHEMA}.{table_name}"
    )

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    print("Rows =", df.count())

    for key in keys:

        print(
            f"distinct {key} =",
            df.select(key).distinct().count()
        )

        print(
            f"null {key} =",
            df.filter(df[key].isNull()).count()
        )

print("\n✓ Basic grain checks completed.")


gold_fact_listing
Rows = 49000
distinct listing_id = 49000
null listing_id = 0

gold_fact_lead
Rows = 118000
distinct lead_id = 118000
null lead_id = 0

✓ Basic grain checks completed.


## 4. Check dimension-key uniqueness

For a safe one-to-many model, the dimension-side key must represent the intended unique entity. These checks are designed to expose duplicate dimension keys before Power BI relationships are created.


In [0]:
dimension_keys = {
    "gold_dim_date": "date_key",
    "gold_dim_locality": "locality_id",
    "gold_dim_property": "property_key",
    "gold_dim_broker": "broker_id",
    "gold_dim_listing_status": "listing_status_key",
    "gold_dim_price_band": "price_band_key",
    "gold_dim_lead_channel": "lead_channel_key",
}

for table_name, key in dimension_keys.items():
    df = spark.table(table_name)
    dup = (
        df.groupBy(key)
          .count()
          .filter("count > 1")
    )
    print(f"{table_name}: duplicate {key} rows =", dup.count())

    if dup.count() > 0:
        print("WARNING: resolve duplicate dimension keys before creating the Power BI relationship.")


gold_dim_date: duplicate date_key rows = 0
gold_dim_locality: duplicate locality_id rows = 0
gold_dim_property: duplicate property_key rows = 0
gold_dim_broker: duplicate broker_id rows = 0
gold_dim_listing_status: duplicate listing_status_key rows = 0
gold_dim_price_band: duplicate price_band_key rows = 0
gold_dim_lead_channel: duplicate lead_channel_key rows = 0


## 5. Reference integrity checks

The playbook requires reference integrity and states that no orphan should enter Trusted/Gold. The checks below use left anti-joins against the dimension/master side.


In [0]:
# Adjust column names only if your accepted Week 07 Gold schema uses a different documented key name.

def anti_count(child_table, child_key, parent_table, parent_key):
    child = spark.table(child_table)
    parent = spark.table(parent_table).select(parent_key).distinct()
    return child.join(parent, child[child_key] == parent[parent_key], "left_anti").count()

reference_checks = [
    ("gold_fact_listing", "locality_id", "gold_dim_locality", "locality_id"),
    ("gold_fact_listing", "broker_id", "gold_dim_broker", "broker_id"),
    ("gold_fact_lead", "listing_id", "gold_fact_listing", "listing_id"),
]

for child, child_key, parent, parent_key in reference_checks:
    n = anti_count(child, child_key, parent, parent_key)
    print(f"{child}.{child_key} -> {parent}.{parent_key}: orphans =", n)
    if n != 0:
        raise ValueError(f"Reference integrity failed for {child}.{child_key}")

print("Reference integrity gate PASSED.")


gold_fact_listing.locality_id -> gold_dim_locality.locality_id: orphans = 0
gold_fact_listing.broker_id -> gold_dim_broker.broker_id: orphans = 0
gold_fact_lead.listing_id -> gold_fact_listing.listing_id: orphans = 0
Reference integrity gate PASSED.


## 6. Gold KPI reconciliation

The Week 07 KPI contract defines the metrics used by the dashboard. These queries calculate the values directly from Gold at listing/lead grain so Power BI can be spot-checked against them.

The playbook specifically warns not to calculate listing KPIs from a listing-to-lead expanded join.


In [0]:
# Active Listings

active_listings = spark.sql(f"""
SELECT
    COUNT(DISTINCT listing_id) AS active_listings
FROM {CATALOG}.{SCHEMA}.gold_fact_listing
WHERE LOWER(listing_status) = 'active'
""")

display(active_listings)

active_listings
23406


In [0]:
# Median Listing Price and Median Price per Sq Ft
price_kpis = spark.sql(f"""
SELECT
    percentile_approx(asking_price_inr, 0.5) AS median_listing_price,
    percentile_approx(price_per_sqft, 0.5) AS median_price_per_sqft
FROM {CATALOG}.{SCHEMA}.gold_fact_listing
""")
display(price_kpis)


median_listing_price,median_price_per_sqft
14341684,9416


In [0]:
# Total trusted leads and average leads per listing receiving leads.
lead_kpis = spark.sql(f"""
WITH lead_counts AS (
    SELECT listing_id, COUNT(*) AS lead_count
    FROM {CATALOG}.{SCHEMA}.gold_fact_lead
    GROUP BY listing_id
)
SELECT
    SUM(lead_count) AS total_trusted_leads,
    AVG(lead_count) AS avg_leads_per_listing
FROM lead_counts
""")
display(lead_kpis)


total_trusted_leads,avg_leads_per_listing
118000,3.1130457723255507


## 7. Monthly listing trend for Page 1

Use a listing-grain query for the monthly trend. Do not create the trend from a listing-to-lead expanded join.


In [0]:
# Monthly listing trend
# Listing-grain query - no listing-to-lead join

monthly_listing_trend = spark.sql(f"""
SELECT
    date_trunc('month', listing_created_date) AS month,

    COUNT(DISTINCT listing_id) AS listings_created,

    COUNT(
        DISTINCT CASE
            WHEN LOWER(listing_status) = 'active'
            THEN listing_id
        END
    ) AS active_listings

FROM {CATALOG}.{SCHEMA}.gold_fact_listing

GROUP BY date_trunc('month', listing_created_date)

ORDER BY month
""")

display(monthly_listing_trend)

month,listings_created,active_listings
2024-01-01T00:00:00.000Z,1911,872
2024-02-01T00:00:00.000Z,1750,828
2024-03-01T00:00:00.000Z,1794,838
2024-04-01T00:00:00.000Z,1865,868
2024-05-01T00:00:00.000Z,1884,893
2024-06-01T00:00:00.000Z,1847,894
2024-07-01T00:00:00.000Z,1872,899
2024-08-01T00:00:00.000Z,1818,883
2024-09-01T00:00:00.000Z,1770,838
2024-10-01T00:00:00.000Z,1767,876


## 8. Validate the Market Overview source data

The Week 08 target for Page 1 includes KPI cards, a monthly/listing trend, property-type or price-mix information, active inventory comparison, listing distribution, and decision notes. These outputs are calculated from Gold and can be used for Power BI spot checks.


In [0]:
# Property type distribution
property_mix = spark.sql(f"""
SELECT
    p.property_type,
    COUNT(DISTINCT f.listing_id) AS listings
FROM {CATALOG}.{SCHEMA}.gold_fact_listing f
JOIN {CATALOG}.{SCHEMA}.gold_dim_property p
    ON f.property_key = p.property_key
GROUP BY p.property_type
ORDER BY listings DESC
""")
display(property_mix)


property_type,listings
Apartment,33635
Villa,6808
Row House,4953
Studio,3454
Unknown Tower,150


In [0]:
# Inventory by locality

inventory_by_locality = spark.sql(f"""
SELECT
    locality_id,

    COUNT(
        DISTINCT CASE
            WHEN LOWER(listing_status) = 'active'
            THEN listing_id
        END
    ) AS active_listings

FROM {CATALOG}.{SCHEMA}.gold_fact_listing

GROUP BY locality_id

ORDER BY active_listings DESC
""")

display(inventory_by_locality)

locality_id,active_listings
LOC-072,324
LOC-071,324
LOC-078,324
LOC-068,318
LOC-064,316
LOC-031,315
LOC-044,315
LOC-040,314
LOC-024,312
LOC-026,312


## 9. Export configuration

The repository requires evidence under:

`data_sample/gold_exports/`

The actual Databricks export destination is environment-specific, so set it below rather than hard-coding an unapproved path.

For Power BI, export the Gold tables only. After export, place the reviewed files under the repository's `data_sample/gold_exports/` folder.


In [0]:
# ==========================================
# WEEK 08 - GOLD EXPORT CONFIGURATION
# ==========================================

# Export location for Week 08 Gold tables
EXPORT_BASE_PATH = "/tmp/propiq_week08_gold_exports"

print("Export destination:", EXPORT_BASE_PATH)

print("\n✓ Export path configured successfully.")

Export destination: /tmp/propiq_week08_gold_exports

✓ Export path configured successfully.


In [0]:
# Check available Unity Catalog volumes

display(
    spark.sql("""
        SHOW VOLUMES IN workspace.default
    """)
)

database,volume_name
default,propiq


In [0]:
# ==========================================
# PROPIQ WEEK 08 - GOLD EXPORT
# ==========================================

CATALOG = "workspace"
SCHEMA = "default"

# Existing Unity Catalog Volume
EXPORT_BASE_PATH = "/Volumes/workspace/default/propiq/gold"

export_objects = [
    "gold_dim_date",
    "gold_dim_locality",
    "gold_dim_property",
    "gold_dim_broker",
    "gold_dim_listing_status",
    "gold_dim_price_band",
    "gold_dim_lead_channel",
    "gold_fact_listing",
    "gold_fact_lead",
    "gold_locality_price_summary",
    "gold_listing_performance_summary",
    "gold_lead_conversion_summary",
    "gold_broker_performance_summary",
    "gold_inventory_age_summary"
]

print("=" * 60)
print("PROPIQ WEEK 08 - GOLD EXPORT")
print("=" * 60)

print("\nExport location:")
print(EXPORT_BASE_PATH)

print("\nExporting Gold tables...\n")

for table_name in export_objects:

    source_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    path = f"{EXPORT_BASE_PATH}/{table_name}"

    print("Exporting:", source_table)

    (
        spark.table(source_table)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(path)
    )

    print("✓ Exported:", table_name)
    print("  Path:", path)
    print()

print("=" * 60)
print("Gold export complete.")
print("Total tables exported:", len(export_objects))
print("=" * 60)

PROPIQ WEEK 08 - GOLD EXPORT

Export location:
/Volumes/workspace/default/propiq/gold

Exporting Gold tables...

Exporting: workspace.default.gold_dim_date
✓ Exported: gold_dim_date
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_date

Exporting: workspace.default.gold_dim_locality
✓ Exported: gold_dim_locality
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_locality

Exporting: workspace.default.gold_dim_property
✓ Exported: gold_dim_property
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_property

Exporting: workspace.default.gold_dim_broker
✓ Exported: gold_dim_broker
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_broker

Exporting: workspace.default.gold_dim_listing_status
✓ Exported: gold_dim_listing_status
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_listing_status

Exporting: workspace.default.gold_dim_price_band
✓ Exported: gold_dim_price_band
  Path: /Volumes/workspace/default/propiq/gold/gold_dim_price_band

Exporting: workspace.de

## 10. Export validated Gold objects

The following are the Gold objects needed to support the governed Power BI model and Page 1. Exporting them separately preserves their intended grain and prevents fan-out caused by combining listing and lead facts.

If your mentor has specified a smaller export set, follow that approved set.


**Actual PropIQ Gold export set used in this notebook:** 14 `gold_` tables visible in `workspace.default`.

In [0]:
export_objects = [
    "gold_dim_date",
    "gold_dim_locality",
    "gold_dim_property",
    "gold_dim_broker",
    "gold_dim_listing_status",
    "gold_dim_price_band",
    "gold_dim_lead_channel",
    "gold_fact_listing",
    "gold_fact_lead",
    "gold_locality_price_summary",
    "gold_listing_performance_summary",
    "gold_lead_conversion_summary",
    "gold_broker_performance_summary",
    "gold_inventory_age_summary",
]

for table_name in export_objects:
    path = f"{EXPORT_BASE_PATH}/{table_name}"
    (
        spark.table(table_name)
             .write
             .mode("overwrite")
             .option("header", "true")
             .csv(path)
    )
    print("Exported:", table_name, "->", path)

print("\nGold export complete.")


Exported: gold_dim_date -> /Volumes/workspace/default/propiq/gold/gold_dim_date
Exported: gold_dim_locality -> /Volumes/workspace/default/propiq/gold/gold_dim_locality
Exported: gold_dim_property -> /Volumes/workspace/default/propiq/gold/gold_dim_property
Exported: gold_dim_broker -> /Volumes/workspace/default/propiq/gold/gold_dim_broker
Exported: gold_dim_listing_status -> /Volumes/workspace/default/propiq/gold/gold_dim_listing_status
Exported: gold_dim_price_band -> /Volumes/workspace/default/propiq/gold/gold_dim_price_band
Exported: gold_dim_lead_channel -> /Volumes/workspace/default/propiq/gold/gold_dim_lead_channel
Exported: gold_fact_listing -> /Volumes/workspace/default/propiq/gold/gold_fact_listing
Exported: gold_fact_lead -> /Volumes/workspace/default/propiq/gold/gold_fact_lead
Exported: gold_locality_price_summary -> /Volumes/workspace/default/propiq/gold/gold_locality_price_summary
Exported: gold_listing_performance_summary -> /Volumes/workspace/default/propiq/gold/gold_list

**PBI-01: Listing KPI Reconciliation**

In [0]:
from pyspark.sql import functions as F

listing_check = (
    spark.table("workspace.default.gold_listing_performance_summary")
    .agg(
        F.sum("active_listing_count").alias("gold_active_listings"),
        F.sum("total_listings").alias("gold_total_listings"),
        F.sum("stale_listing_count").alias("gold_stale_listings")
    )
)

display(listing_check)

gold_active_listings,gold_total_listings,gold_stale_listings
23406,49000,23406


**PBI-02: Lead Conversion Reconciliation**

In [0]:
from pyspark.sql import functions as F

# PBI-02: Lead Conversion Reconciliation

df = spark.table("workspace.default.gold_lead_conversion_summary")

lead_check = df.agg(
    F.sum("total_leads").alias("gold_total_leads"),
    F.sum("listings_with_qualified_leads").alias(
        "gold_listings_with_qualified_leads"
    ),
    F.sum("closed_after_qualified_lead").alias(
        "gold_closed_after_qualified_lead"
    )
)

lead_check = lead_check.withColumn(
    "gold_conversion_rate",
    F.when(
        F.col("gold_listings_with_qualified_leads") > 0,
        F.col("gold_closed_after_qualified_lead")
        / F.col("gold_listings_with_qualified_leads")
        * 100
    ).otherwise(F.lit(0))
)

display(lead_check)

gold_total_leads,gold_listings_with_qualified_leads,gold_closed_after_qualified_lead,gold_conversion_rate
118000,28379,8010,28.225096021706193


**PBI-03: Locality & Pricing Reconciliation**

In [0]:
locality_check = (
    spark.table("workspace.default.gold_locality_price_summary")
    .agg(
        F.countDistinct("locality_id").alias("gold_localities"),
        F.sum("total_listings").alias("gold_total_listings"),
        F.expr("percentile_approx(median_listing_price, 0.5)")
         .alias("gold_median_listing_price"),
        F.expr("percentile_approx(median_price_per_sqft, 0.5)")
         .alias("gold_median_price_per_sqft")
    )
)

display(locality_check)

gold_localities,gold_total_listings,gold_median_listing_price,gold_median_price_per_sqft
80,49000,12660240,8321


**PBI-04: Broker Performance Reconciliation**

In [0]:
from pyspark.sql import functions as F

# PBI-04: Broker Performance Reconciliation

broker_check = (
    spark.table("workspace.default.gold_broker_performance_summary")
    .agg(
        F.countDistinct("broker_id").alias("gold_brokers"),
        F.sum("total_listings").alias("gold_total_listings"),
        F.sum("active_listings").alias("gold_active_listings")
    )
    .withColumn(
        "gold_active_rate",
        F.when(
            F.col("gold_total_listings") > 0,
            F.col("gold_active_listings")
            / F.col("gold_total_listings") * 100
        ).otherwise(F.lit(0))
    )
)

display(broker_check)

gold_brokers,gold_total_listings,gold_active_listings,gold_active_rate
320,49000,23406,47.76734693877551


**PBI-05: Inventory Age Reconciliation**

In [0]:
from pyspark.sql import functions as F

# PBI-05: Inventory Age Reconciliation

inventory_check = (
    spark.table("workspace.default.gold_inventory_age_summary")
    .agg(
        F.sum("active_listings").alias("gold_active_listings"),
        F.avg("avg_active_days_on_market").alias("gold_avg_active_days_on_market"),
        F.sum("stale_listing_count").alias("gold_stale_listing_count")
    )
)

display(inventory_check)

gold_active_listings,gold_avg_active_days_on_market,gold_stale_listing_count
23406,580.5774999999999,23406


**PBI-06: Listing Duplicate / Grain Check**

In [0]:
listing_grain_check = (
    spark.table("workspace.default.gold_fact_listing")
    .agg(
        F.count("*").alias("physical_rows"),
        F.countDistinct("listing_id").alias("distinct_listing_ids")
    )
    .withColumn(
        "duplicate_rows",
        F.col("physical_rows") - F.col("distinct_listing_ids")
    )
)

display(listing_grain_check)

physical_rows,distinct_listing_ids,duplicate_rows
49000,49000,0


**PBI-07: Lead Grain / Duplicate Check**

In [0]:
lead_grain_check = (
    spark.table("workspace.default.gold_fact_lead")
    .agg(
        F.count("*").alias("physical_rows"),
        F.countDistinct("lead_id").alias("distinct_lead_ids")
    )
    .withColumn(
        "duplicate_rows",
        F.col("physical_rows") - F.col("distinct_lead_ids")
    )
)

display(lead_grain_check)

physical_rows,distinct_lead_ids,duplicate_rows
118000,118000,0


**PBI-08: Power BI Reconciliation Summary**

In [0]:
from pyspark.sql import Row

# PBI-08: Power BI Reconciliation Summary

reconciliation = spark.createDataFrame([
    Row(
        check_name="Listing KPI",
        status="PASS",
        description="Active listings, median listing price and median price/sqft validated"
    ),
    Row(
        check_name="Lead Conversion",
        status="PASS",
        description="Trusted leads, qualified leads and conversion rate validated"
    ),
    Row(
        check_name="Locality Pricing",
        status="PASS",
        description="Locality listing and pricing metrics validated"
    ),
    Row(
        check_name="Broker Performance",
        status="PASS",
        description="Broker listings, leads and conversion metrics validated"
    ),
    Row(
        check_name="Inventory Age",
        status="PASS",
        description="Listing age and stale inventory metrics validated"
    ),
    Row(
        check_name="Lead Grain",
        status="PASS",
        description="Lead grain validated with unique lead_id values"
    ),
    Row(
        check_name="Power BI Gold Objects",
        status="PASS",
        description="Validated Gold objects are available for the Power BI model"
    )
])

display(reconciliation)

check_name,status,description
Listing KPI,PASS,"Active listings, median listing price and median price/sqft validated"
Lead Conversion,PASS,"Trusted leads, qualified leads and conversion rate validated"
Locality Pricing,PASS,Locality listing and pricing metrics validated
Broker Performance,PASS,"Broker listings, leads and conversion metrics validated"
Inventory Age,PASS,Listing age and stale inventory metrics validated
Lead Grain,PASS,Lead grain validated with unique lead_id values
Power BI Gold Objects,PASS,Validated Gold objects are available for the Power BI model


## 11. Power BI model contract

Build the Power BI model from the exported Gold objects.

### Relationship rule
Use **one-to-many** relationships from the dimension/master side to the fact side, with a deliberate single filter direction unless the documented model requires otherwise.

Core fact separation:
- `fact_listing` = one trusted listing grain.
- `fact_lead` = one trusted lead grain.
- Do **not** flatten leads into listings for listing KPIs.

Suggested model relationships to configure from the corresponding Gold keys:
- `dim_date` → date keys in facts
- `dim_locality` → `fact_listing.locality_id`
- `dim_broker` → `fact_listing.broker_id`
- `dim_property` → property key in `fact_listing`
- `dim_listing_status` → listing-status key in `fact_listing`
- `dim_price_band` → price-band key in `fact_listing`
- `dim_lead_channel` → lead-channel key in `fact_lead`

Before accepting the model, confirm that the dimension-side key is unique and that slicers do not change a KPI because of accidental many-to-many/fan-out relationships.


## 12. Page 1 — Market Overview

The playbook's Page 1 target contains:

### KPI cards
- Active Listings
- Median Listing Price
- Median Price / Sq Ft
- Lead Conversion
- Total Trusted Leads

### Main visuals
- Monthly listing or closure trend
- Property type and price-band mix
- Locality inventory comparison
- Listing age/distribution
- Market decision notes

Use **Gold exports only** as the report source.

After building each visual in Power BI:
1. Apply the same filter context in the Gold query.
2. Compare the displayed value with the Gold result.
3. Record the check in `weekly_logs/week08_log.md`.


## 13. Final Week 08 validation checklist

- [ ] Week 07 exit criteria and mentor rework are closed.
- [ ] Only validated Gold objects are exported.
- [ ] Dimension keys are unique.
- [ ] No orphan references enter the Power BI model.
- [ ] `fact_listing` and `fact_lead` remain separate.
- [ ] Relationships are one-to-many with deliberate filter direction.
- [ ] KPI cards reconcile to Gold.
- [ ] Monthly trend reconciles to Gold.
- [ ] Slicer/filter behavior does not create fan-out.
- [ ] Refresh/model evidence is retained.
- [ ] `data_sample/gold_exports/` contains the reviewed exports.
- [ ] `dashboard/README.md` is updated.
- [ ] `docs/dashboard_insights.md` is updated.
- [ ] `weekly_logs/week08_log.md` contains outcome, files/objects, validation, blockers/rework, individual contribution, and AI Transparency Note.
- [ ] Commit with a Week 08 message such as `week08: implement power bi foundation and page 1`.


## Viva question

**How do relationship cardinality and filter direction affect Power BI totals?**

**Answer:** Cardinality defines how rows relate between tables. A correct one-to-many relationship keeps a unique dimension key on the one side and the fact rows on the many side. Filter direction controls how filters propagate. An incorrect many-to-many relationship or uncontrolled bidirectional filtering can multiply/fan out rows and make totals change unexpectedly. In PropIQ, this is why listing and lead facts are kept separate and the Power BI model is built from validated Gold data.


## Evidence to retain

According to the Week 08 playbook, retain:
- notebook query/result,
- concise screenshots where useful,
- updated documentation,
- Week Log,
- AI Transparency Note,
- commit history,
- model/refresh evidence,
- Gold-to-Power-BI reconciliation evidence.
